# VishGuard — Colab setup (T1.5)

Self-contained setup cell. Colab notebooks have isolated runtimes — no
notebook in this project assumes another has already run. Copy the setup
cell below into every new notebook.

Two ways to get the repo onto the runtime:

1. **GitHub clone** (preferred once the repo is public): set `REPO_URL`.
2. **Drive copy**: copy the repo into `MyDrive/vishguard/` and the cell
   will `rsync` it into `/content/vishguard`.

Acceptance: the final cell prints `vishguard 0.1.0`.

In [ ]:
# --- VishGuard setup cell (copy into every notebook) ----------------
import os, sys, subprocess, shutil
from pathlib import Path

REPO_URL = os.environ.get('VISHGUARD_REPO_URL', '')  # set to https://github.com/<you>/vishguard.git once pushed
DRIVE_SRC = '/content/drive/MyDrive/vishguard'  # fallback if no GitHub URL yet
REPO_DIR = Path('/content/vishguard')
ON_COLAB = 'google.colab' in sys.modules or Path('/content').exists()

def sh(cmd, check=True):
    print('$', cmd)
    subprocess.run(cmd, shell=True, check=check)

if ON_COLAB:
    # 1. Mount Drive (no-op if already mounted)
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except Exception as e:
        print('Drive mount skipped:', e)

    # 2. Materialize the repo into /content/vishguard
    if REPO_URL:
        if REPO_DIR.exists():
            sh(f'cd {REPO_DIR} && git pull --ff-only')
        else:
            sh(f'git clone {REPO_URL} {REPO_DIR}')
    elif Path(DRIVE_SRC).exists():
        REPO_DIR.mkdir(parents=True, exist_ok=True)
        sh(f'rsync -a --delete --exclude .venv --exclude __pycache__ {DRIVE_SRC}/ {REPO_DIR}/')
    else:
        raise RuntimeError('Set VISHGUARD_REPO_URL or copy the repo to MyDrive/vishguard/')

    # 3. Install package + deps. bitsandbytes is GPU-only; skip on CPU runtimes.
    sh(f'pip install -q -e {REPO_DIR}')
    sh(f'pip install -q -r {REPO_DIR}/requirements.txt')
    try:
        import torch
        if torch.cuda.is_available():
            sh(f'pip install -q -r {REPO_DIR}/requirements-gpu.txt')
    except Exception as e:
        print('bitsandbytes skipped:', e)

    # 4. sys.path (src-layout backup in case editable install didn't register)
    src = str(REPO_DIR / 'src')
    if src not in sys.path:
        sys.path.insert(0, src)

# 5. Smoke test
import vishguard
print('vishguard', vishguard.__version__)

## Next steps

- `01_spike_antiSpoof.ipynb` — T1.1, MelodyMachine/Deepfake-audio-detection-V2 smoke test
- `02_spike_llmJson.ipynb` — T1.2, Qwen2.5-3B-Instruct JSON reliability
- `03_spike_whisperWer.ipynb` — T1.3, Whisper WER floor on clean + noisy speech